# 📦 Phase 1 — Data Cleaning & Preprocessing
**Project**: Blinkit Supply Chain & Inventory Optimization  
**Author**: Rishi Verma  
**Date**: 2024

---
### 🎯 Goals:
- Load all 8 datasets
- Inspect shape, dtypes, nulls, duplicates
- Clean and standardize each dataset
- Merge into master dataframes
- Export cleaned CSVs for next phases
---

## 1️⃣ Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('✅ Libraries loaded successfully!')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy  version: {np.__version__}')

## 2️⃣ Load All Datasets

In [ ]:
# ─── UPDATE THIS PATH to your data folder ───
DATA_PATH = r"d:\blinkit project"
CLEANED_PATH = r"../data/cleaned"
os.makedirs(CLEANED_PATH, exist_ok=True)

# Load all datasets
products    = pd.read_csv(f"{DATA_PATH}/blinkit_products.csv")
inventory   = pd.read_csv(f"{DATA_PATH}/blinkit_inventory.csv")
orders      = pd.read_csv(f"{DATA_PATH}/blinkit_orders.csv")
order_items = pd.read_csv(f"{DATA_PATH}/blinkit_order_items.csv")
customers   = pd.read_csv(f"{DATA_PATH}/blinkit_customers.csv")
delivery    = pd.read_csv(f"{DATA_PATH}/blinkit_delivery_performance.csv")
feedback    = pd.read_csv(f"{DATA_PATH}/blinkit_customer_feedback.csv")
marketing   = pd.read_csv(f"{DATA_PATH}/blinkit_marketing_performance.csv")

datasets = {
    'products'   : products,
    'inventory'  : inventory,
    'orders'     : orders,
    'order_items': order_items,
    'customers'  : customers,
    'delivery'   : delivery,
    'feedback'   : feedback,
    'marketing'  : marketing
}

print('✅ All datasets loaded!\n')
print(f'{"Dataset":<20} {"Rows":>8} {"Cols":>6}')
print('-' * 36)
for name, df in datasets.items():
    print(f'{name:<20} {df.shape[0]:>8,} {df.shape[1]:>6}')

## 3️⃣ Quick Overview of Each Dataset

In [ ]:
def dataset_overview(df, name):
    """Print a detailed overview of a dataframe."""
    print(f"\n{'='*55}")
    print(f"  📋 {name.upper()}")
    print(f"{'='*55}")
    print(f"  Shape      : {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"  Duplicates : {df.duplicated().sum():,}")
    print(f"\n  Null Values:")
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if nulls.empty:
        print("    ✅ No null values found!")
    else:
        for col, cnt in nulls.items():
            print(f"    ⚠️  {col}: {cnt} nulls ({cnt/len(df)*100:.1f}%)")
    print(f"\n  Data Types:")
    for col, dtype in df.dtypes.items():
        print(f"    {col:<35} {str(dtype)}")

for name, df in datasets.items():
    dataset_overview(df, name)

## 4️⃣ Clean: Products

In [ ]:
print("🧹 Cleaning: blinkit_products.csv")
prod = products.copy()

# Check
print(f"Before: {prod.shape}")
print(prod.head(3))

# Drop duplicates
prod.drop_duplicates(inplace=True)

# Strip whitespace from string columns
str_cols = prod.select_dtypes(include='object').columns
prod[str_cols] = prod[str_cols].apply(lambda x: x.str.strip())

# Standardize category names
prod['category'] = prod['category'].str.title()
prod['brand'] = prod['brand'].str.title()

# Feature Engineering
# Stock range (buffer between min and max)
prod['stock_range'] = prod['max_stock_level'] - prod['min_stock_level']

# Price discount % from MRP
prod['discount_pct'] = ((prod['mrp'] - prod['price']) / prod['mrp'] * 100).round(2)

# Perishable flag (shelf_life <= 7 days)
prod['is_perishable'] = prod['shelf_life_days'].apply(lambda x: 'Yes' if x <= 7 else 'No')

# Revenue potential = price × max_stock_level
prod['revenue_potential'] = (prod['price'] * prod['max_stock_level']).round(2)

print(f"\nAfter : {prod.shape}")
print(f"\n✅ Products cleaned!")
print(prod[['product_name','category','price','mrp','discount_pct','is_perishable','stock_range']].head(8))

## 5️⃣ Clean: Inventory

In [ ]:
print("🧹 Cleaning: blinkit_inventory.csv")
inv = inventory.copy()

# Parse date
inv['date'] = pd.to_datetime(inv['date'], dayfirst=True)

# Drop duplicates
inv.drop_duplicates(inplace=True)

# Remove rows where stock_received is negative (data quality issue)
inv = inv[inv['stock_received'] >= 0]
inv = inv[inv['damaged_stock'] >= 0]

# Ensure damaged_stock never exceeds stock_received
inv = inv[inv['damaged_stock'] <= inv['stock_received']]

# Feature Engineering
inv['good_stock'] = inv['stock_received'] - inv['damaged_stock']
inv['damage_rate'] = (inv['damaged_stock'] / inv['stock_received'].replace(0, np.nan) * 100).round(2)
inv['year']  = inv['date'].dt.year
inv['month'] = inv['date'].dt.month
inv['month_name'] = inv['date'].dt.strftime('%b')
inv['week']  = inv['date'].dt.isocalendar().week.astype(int)
inv['day_of_week'] = inv['date'].dt.day_name()

print(f"Shape: {inv.shape}")
print(f"Date range: {inv['date'].min()} → {inv['date'].max()}")
print(f"Avg damage rate: {inv['damage_rate'].mean():.2f}%")
print(f"\n✅ Inventory cleaned!")
print(inv.head(5))

## 6️⃣ Clean: Orders

In [ ]:
print("🧹 Cleaning: blinkit_orders.csv")
ord_df = orders.copy()

# Parse datetimes
ord_df['order_date']             = pd.to_datetime(ord_df['order_date'])
ord_df['promised_delivery_time'] = pd.to_datetime(ord_df['promised_delivery_time'])
ord_df['actual_delivery_time']   = pd.to_datetime(ord_df['actual_delivery_time'])

# Drop duplicates
ord_df.drop_duplicates(subset='order_id', inplace=True)

# Strip string columns
str_cols = ord_df.select_dtypes(include='object').columns
ord_df[str_cols] = ord_df[str_cols].apply(lambda x: x.str.strip())

# Feature Engineering
ord_df['order_year']    = ord_df['order_date'].dt.year
ord_df['order_month']   = ord_df['order_date'].dt.month
ord_df['order_month_name'] = ord_df['order_date'].dt.strftime('%b')
ord_df['order_day']     = ord_df['order_date'].dt.day_name()
ord_df['order_hour']    = ord_df['order_date'].dt.hour

# Delivery delay in minutes
ord_df['delay_minutes'] = (
    (ord_df['actual_delivery_time'] - ord_df['promised_delivery_time'])
    .dt.total_seconds() / 60
).round(2)

# Is late flag
ord_df['is_late'] = ord_df['delay_minutes'].apply(lambda x: 1 if x > 0 else 0)

print(f"Shape: {ord_df.shape}")
print(f"Date range: {ord_df['order_date'].min().date()} → {ord_df['order_date'].max().date()}")
print(f"Late orders: {ord_df['is_late'].sum()} ({ord_df['is_late'].mean()*100:.1f}%)")
print(f"Total Revenue: ₹{ord_df['order_total'].sum():,.2f}")
print(f"\n✅ Orders cleaned!")
print(ord_df[['order_id','order_date','order_total','delivery_status','delay_minutes','is_late']].head(5))

## 7️⃣ Clean: Order Items

In [ ]:
print("🧹 Cleaning: blinkit_order_items.csv")
oi = order_items.copy()

# Drop duplicates
oi.drop_duplicates(inplace=True)

# Remove negative or zero values
oi = oi[(oi['quantity'] > 0) & (oi['unit_price'] > 0)]

# Feature Engineering
oi['line_total'] = (oi['quantity'] * oi['unit_price']).round(2)

print(f"Shape: {oi.shape}")
print(f"Unique products ordered: {oi['product_id'].nunique()}")
print(f"Total line value: ₹{oi['line_total'].sum():,.2f}")
print(f"\n✅ Order items cleaned!")
print(oi.head(5))

## 8️⃣ Clean: Customers

In [ ]:
print("🧹 Cleaning: blinkit_customers.csv")
cust = customers.copy()

cust.drop_duplicates(subset='customer_id', inplace=True)
cust['registration_date'] = pd.to_datetime(cust['registration_date'])

str_cols = cust.select_dtypes(include='object').columns
cust[str_cols] = cust[str_cols].apply(lambda x: x.str.strip())
cust['customer_segment'] = cust['customer_segment'].str.title()

# Feature: Customer tenure in days
cust['tenure_days'] = (pd.Timestamp.now() - cust['registration_date']).dt.days

print(f"Shape: {cust.shape}")
print(f"Segments: {cust['customer_segment'].value_counts().to_dict()}")
print(f"\n✅ Customers cleaned!")
print(cust.head(3))

## 9️⃣ Clean: Delivery Performance

In [ ]:
print("🧹 Cleaning: blinkit_delivery_performance.csv")
deliv = delivery.copy()

deliv.drop_duplicates(subset='order_id', inplace=True)
str_cols = deliv.select_dtypes(include='object').columns
deliv[str_cols] = deliv[str_cols].apply(lambda x: x.str.strip())

# Remove outlier delivery times (> 120 min is anomaly for quick-commerce)
deliv = deliv[deliv['delivery_time_minutes'] > 0]

# Category for delivery speed
deliv['delivery_speed'] = pd.cut(
    deliv['delivery_time_minutes'],
    bins=[0, 15, 30, 60, float('inf')],
    labels=['⚡ Ultra Fast (<15m)', '✅ Fast (15-30m)', '⚠️ Slow (30-60m)', '❌ Very Slow (>60m)']
)

print(f"Shape: {deliv.shape}")
print(f"Avg delivery time: {deliv['delivery_time_minutes'].mean():.1f} min")
print(f"\nDelivery speed breakdown:")
print(deliv['delivery_speed'].value_counts())
print(f"\n✅ Delivery data cleaned!")

## 🔟 Clean: Feedback & Marketing

In [ ]:
# Feedback
print("🧹 Cleaning: blinkit_customer_feedback.csv")
fb = feedback.copy()
fb.drop_duplicates(inplace=True)
fb['feedback_date'] = pd.to_datetime(fb['feedback_date'])
str_cols = fb.select_dtypes(include='object').columns
fb[str_cols] = fb[str_cols].apply(lambda x: x.str.strip())
fb['sentiment'] = fb['sentiment'].str.title()
fb['rating'] = fb['rating'].clip(1, 5)  # ratings must be 1–5
print(f"Shape: {fb.shape}")
print(f"Avg rating: {fb['rating'].mean():.2f}")
print(f"Sentiment: {fb['sentiment'].value_counts().to_dict()}")

# Marketing
print("\n🧹 Cleaning: blinkit_marketing_performance.csv")
mkt = marketing.copy()
mkt.drop_duplicates(inplace=True)
mkt['date'] = pd.to_datetime(mkt['date'])
str_cols = mkt.select_dtypes(include='object').columns
mkt[str_cols] = mkt[str_cols].apply(lambda x: x.str.strip())
mkt['ctr'] = (mkt['clicks'] / mkt['impressions'].replace(0, np.nan) * 100).round(2)
mkt['conversion_rate'] = (mkt['conversions'] / mkt['clicks'].replace(0, np.nan) * 100).round(2)
print(f"Shape: {mkt.shape}")
print(f"Avg ROAS: {mkt['roas'].mean():.2f}")

## 1️⃣1️⃣ Create Master Dataset — Orders + Items + Products

In [ ]:
print("🔗 Building master sales dataset...")

# Merge order_items ← products
master_sales = oi.merge(prod, on='product_id', how='left')

# Merge ← orders (for date, store, customer info)
master_sales = master_sales.merge(
    ord_df[['order_id','customer_id','order_date','order_month','order_month_name',
            'order_day','order_hour','delivery_status','store_id','is_late','delay_minutes']],
    on='order_id', how='left'
)

# Merge ← customers (segment info)
master_sales = master_sales.merge(
    cust[['customer_id','area','customer_segment','total_orders','avg_order_value']],
    on='customer_id', how='left'
)

print(f"Master Sales Shape: {master_sales.shape}")
print(f"Columns: {list(master_sales.columns)}")
print(master_sales.head(3))

## 1️⃣2️⃣ Create Inventory Master — Inventory + Products

In [ ]:
print("🔗 Building master inventory dataset...")

master_inventory = inv.merge(
    prod[['product_id','product_name','category','price','mrp',
          'min_stock_level','max_stock_level','shelf_life_days',
          'is_perishable','margin_percentage','revenue_potential']],
    on='product_id', how='left'
)

print(f"Master Inventory Shape: {master_inventory.shape}")
print(master_inventory.head(3))

## 1️⃣3️⃣ Export Cleaned Datasets

In [ ]:
# Export all cleaned dataframes
exports = {
    'products_cleaned.csv'        : prod,
    'inventory_cleaned.csv'       : inv,
    'orders_cleaned.csv'          : ord_df,
    'order_items_cleaned.csv'     : oi,
    'customers_cleaned.csv'       : cust,
    'delivery_cleaned.csv'        : deliv,
    'feedback_cleaned.csv'        : fb,
    'marketing_cleaned.csv'       : mkt,
    'master_sales.csv'            : master_sales,
    'master_inventory.csv'        : master_inventory
}

for filename, df in exports.items():
    path = os.path.join(CLEANED_PATH, filename)
    df.to_csv(path, index=False)
    print(f"✅ Saved: {filename} ({df.shape[0]:,} rows × {df.shape[1]} cols)")

print("\n🎉 Phase 1 Complete! All cleaned datasets exported to data/cleaned/")

## 1️⃣4️⃣ Phase 1 Summary

In [ ]:
print("=" * 55)
print("📋 PHASE 1 — DATA CLEANING SUMMARY")
print("=" * 55)
print(f"  Total unique products   : {prod['product_id'].nunique():,}")
print(f"  Product categories      : {prod['category'].nunique()}")
print(f"  Perishable items        : {(prod['is_perishable']=='Yes').sum()} / {len(prod)}")
print(f"")
print(f"  Inventory records       : {inv.shape[0]:,}")
print(f"  Date range              : {inv['date'].min().date()} → {inv['date'].max().date()}")
print(f"  Avg daily damage rate   : {inv['damage_rate'].mean():.2f}%")
print(f"")
print(f"  Total orders            : {ord_df.shape[0]:,}")
print(f"  Total revenue           : ₹{ord_df['order_total'].sum():,.0f}")
print(f"  Late deliveries         : {ord_df['is_late'].sum()} ({ord_df['is_late'].mean()*100:.1f}%)")
print(f"")
print(f"  Master sales rows       : {master_sales.shape[0]:,}")
print(f"  Master inventory rows   : {master_inventory.shape[0]:,}")
print("=" * 55)
print("  ✅ Ready for Phase 2 — EDA!")
print("=" * 55)